# EverPeak Retail — Análisis de calidad de datos y EDA

**Contexto:** SilverBasket Retail Group planea incorporar a EverPeak Holdings. Antes de cerrar la adquisición, el equipo de Estrategia e Integración necesita evaluar la calidad y la estructura de los datos transaccionales de EverPeak (2024).

**Objetivos**
1. Diagnosticar la estructura del dataset y los valores ausentes o inválidos.
2. Construir un pipeline de limpieza reutilizable con funciones de Python.
3. Describir las distribuciones de las variables clave (EDA).
4. Detectar y tratar valores atípicos (regla IQR).
5. Segmentar clientes y redactar un resumen estadístico para el negocio.

**Autor:** Germán Ricardo Villamizar · Bootcamp Analista de Datos — TripleTen, Sprint 7

## 1. Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

URL = "https://practicum-content.s3.us-west-1.amazonaws.com/datasets/everpeak_retail.csv"
df_raw = pd.read_csv(URL)
df_raw.head()

## 2. Diagnóstico inicial

In [ ]:
df_raw.info()

In [ ]:
# Proporción de valores ausentes por columna (%)
(df_raw.isna().mean() * 100).round(2).sort_values(ascending=False)

In [ ]:
# Cardinalidad: número de valores únicos por columna
df_raw.nunique().sort_values()

In [ ]:
# Revisión de columnas categóricas: códigos inválidos como '?'
for col in ['product_category', 'payment_method', 'city', 'state']:
    print(f"--- {col} ---")
    print(df_raw[col].value_counts(dropna=False), "\n")

In [ ]:
df_raw.describe().T

**Señales de alerta a revisar en la limpieza:**
- `product_category` contiene el código inválido `'?'`.
- `city`, `state`, `customer_age` y `order_date` tienen valores ausentes.
- `customer_age` incluye valores imposibles (por ejemplo `-999`, usado como marcador de "desconocido").
- `order_date` está como texto y contiene fechas fuera del año analizado (2024).
- `quantity` tiene muchos registros en `0`, lo que no es coherente con una orden con `order_value` positivo.
- `price` y `order_value` muestran colas largas que sugieren valores atípicos.

## 3. Funciones de limpieza

In [ ]:
def limpiar_categoria(df, col='product_category', invalidos=('?', '', 'unknown')):
    """Reemplaza códigos inválidos por 'Unknown'."""
    df = df.copy()
    df[col] = df[col].replace(list(invalidos), np.nan).fillna('Unknown')
    return df


def limpiar_fechas(df, col='order_date', anio=2024):
    """Convierte a datetime y marca como NaT las fechas fuera del año de análisis."""
    df = df.copy()
    df[col] = pd.to_datetime(df[col], errors='coerce')
    df.loc[df[col].dt.year != anio, col] = pd.NaT
    return df


def limpiar_edad(df, col='customer_age', minimo=18, maximo=100):
    """Convierte edades imposibles (p. ej. -999) en NaN."""
    df = df.copy()
    df.loc[~df[col].between(minimo, maximo), col] = np.nan
    return df


def limpiar_ubicacion(df, cols=('city', 'state')):
    """Rellena ciudad/estado ausentes con 'Unknown'."""
    df = df.copy()
    for c in cols:
        df[c] = df[c].fillna('Unknown')
    return df


def marcar_cantidad_invalida(df, col='quantity'):
    """Crea una bandera para cantidades <= 0 (no se eliminan para no perder el order_value)."""
    df = df.copy()
    df['quantity_invalida'] = df[col] <= 0
    return df

## 4. Pipeline completo de limpieza

In [ ]:
def pipeline_limpieza(df):
    pasos = [limpiar_categoria, limpiar_fechas, limpiar_edad,
             limpiar_ubicacion, marcar_cantidad_invalida]
    for paso in pasos:
        df = paso(df)
    return df.drop_duplicates()

df = pipeline_limpieza(df_raw)
print("Filas antes:", len(df_raw), "| Filas después:", len(df))
df.isna().sum()

In [ ]:
# Verificación: comparación de ausentes antes y después
pd.DataFrame({'antes': df_raw.isna().sum(), 'despues': df.isna().sum()})

## 5. Describiendo distribuciones

In [ ]:
# Medidas estadísticas en columnas numéricas
num_cols = ['price', 'quantity', 'order_value', 'customer_age']
resumen = df[num_cols].agg(['mean', 'median', 'std', 'min', 'max']).T
resumen['media/mediana'] = (resumen['mean'] / resumen['median']).round(2)
resumen.round(2)

In [ ]:
# Medidas en columnas categóricas: frecuencias relativas (%)
for col in ['product_category', 'payment_method', 'state']:
    print(f"--- {col} ---")
    print((df[col].value_counts(normalize=True) * 100).round(1), "\n")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['price', 'order_value', 'customer_age']):
    df[col].dropna().plot(kind='hist', bins=50, ax=ax, edgecolor='white')
    ax.set_title(f'Histograma de {col}')
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df.boxplot(column='order_value', by='product_category', ax=axes[0], rot=45)
axes[0].set_title('order_value por categoría')
df.boxplot(column='price', ax=axes[1])
axes[1].set_title('Boxplot de price')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 6. Valores atípicos (regla IQR)

In [ ]:
def limites_iqr(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

for col in ['price', 'order_value']:
    li, ls = limites_iqr(df[col])
    n_out = (~df[col].between(li, ls)).sum()
    print(f"{col}: límites [{li:,.0f}, {ls:,.0f}] -> {n_out} atípicos ({n_out/len(df):.1%})")

In [ ]:
# Tratamiento: se marcan los atípicos y se crea una versión 'winsorizada' para análisis de tendencia central
for col in ['price', 'order_value']:
    li, ls = limites_iqr(df[col])
    df[f'{col}_atipico'] = ~df[col].between(li, ls)
    df[f'{col}_cap'] = df[col].clip(lower=max(li, 0), upper=ls)

df[['price', 'price_cap', 'order_value', 'order_value_cap']].describe().round(1)

**Decisión:** los atípicos no se eliminan porque pueden ser compras mayoristas reales. Se conservan con una bandera
y se usan las columnas `_cap` solo cuando se necesita una medida central robusta.

## 7. Segmentación de clientes

In [ ]:
clientes = (df.groupby('customer_id')
              .agg(ordenes=('order_id', 'count'),
                   gasto_total=('order_value', 'sum'),
                   edad=('customer_age', 'median'))
              .reset_index())

def segmentar(gasto, p50, p90):
    if gasto >= p90:
        return 'VIP'
    elif gasto >= p50:
        return 'Medio'
    else:
        return 'Bajo'

p50, p90 = clientes['gasto_total'].quantile([0.5, 0.9])
clientes['segmento'] = clientes['gasto_total'].apply(segmentar, args=(p50, p90))

seg = clientes.groupby('segmento').agg(clientes=('customer_id', 'count'),
                                       gasto_medio=('gasto_total', 'mean'),
                                       ordenes_medias=('ordenes', 'mean'))
seg['% ingresos'] = (clientes.groupby('segmento')['gasto_total'].sum() / clientes['gasto_total'].sum() * 100)
seg.round(1)

## 8. Resumen estadístico para el negocio

> Completa esta sección con las cifras obtenidas al ejecutar el notebook.

- **Calidad de datos:** el dataset tiene un nivel bajo de valores ausentes, concentrado en `customer_age`, `city`/`state` y `order_date`. Se detectaron códigos inválidos (`'?'`, `-999`) y fechas fuera de 2024, tratados en el pipeline.
- **Cantidad (`quantity`):** una proporción alta de órdenes registra `0` unidades pese a tener valor de venta; se recomienda validar este campo con el equipo de sistemas de EverPeak antes de usarlo en análisis de volumen.
- **Distribuciones:** `price` y `order_value` son asimétricas a la derecha (media > mediana), por lo que la mediana es la medida central más representativa.
- **Atípicos:** existe un grupo de órdenes de valor muy alto; se conservaron y marcaron para no sesgar los promedios.
- **Segmentos:** el 10 % de clientes VIP concentra una parte desproporcionada de los ingresos — prioridad para retención tras la integración.

**Próximos pasos:** validar `quantity` y las fechas con la fuente, y repetir el análisis con datos de 2025.